In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *


In [0]:
dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
catalog_name = dbutils.widgets.get("catalog_name")

In [0]:
%sql
use catalog ecommerce;

In [0]:
path = f"/Volumes/{catalog_name}/raw/raw_landing/historical-full-load/order_items/landing/"
bronze_checkpoint_path = f"/Volumes/ecommerce/raw/raw_landing/checkpoint/bronze/fact_order_items/"


In [0]:
spark.readStream \
 .format("cloudFiles") \
 .option("cloudFiles.format", "csv")  \
 .option("cloudFiles.schemaLocation", bronze_checkpoint_path) \
 .option("cloudFiles.schemaEvolutionMode", "rescue") \
 .option("header", "true") \
 .option("cloudFiles.inferColumnTypes", "true") \
 .option("rescuedDataColumn", "_rescued_data") \
 .option("cloudFiles.includeExistingFiles", "true")  \
 .option("pathGlobFilter", "*.csv") \
 .load(path) \
 .withColumn("ingest_timestamp", F.current_timestamp()) \
 .withColumn("source_file", F.col("_metadata.file_path")) \
 .writeStream \
 .outputMode("append") \
 .option("checkpointLocation", bronze_checkpoint_path) \
 .trigger(availableNow=True) \
 .toTable(f"{catalog_name}.bronze.brz_order_items") \
 .awaitTermination()

In [0]:
%sql
SELECT max(dt) FROM ecommerce.bronze.brz_order_items

In [0]:
%sql
SELECT min(dt) FROM ecommerce.bronze.brz_order_items

In [0]:
%sql
SELECT count(*) FROM ecommerce.bronze.brz_order_items